# Create Graph Dataset

In [24]:
import sys
import os
import pickle as pkl
import pandas as pd
import torch

path = os.path.join('..', '.')
if path not in sys.path:
    sys.path.append(os.path.abspath(path))

from src.protein_graph import pncaGraph

from tqdm import tqdm

import warnings
warnings.filterwarnings('ignore')

In [36]:
train_seqs = pd.read_csv('../data/real_train_sequences.csv')
test_seqs = pd.read_csv('../data/real_test_sequences.csv')

clustered_train_seqs = pd.read_csv('../data/clustered_train_sequences.csv')
clustered_test_seqs = pd.read_csv('../data/clustered_test_sequences.csv')

codon_train_seqs = pd.read_csv('../data/codon_train_sequences.csv')
codon_test_seqs = pd.read_csv('../data/codon_test_sequences.csv')

In [37]:
split_method = 'codon' # 'clustered' or 'codon'

### Create graphs and corresponding Data objects

Using AlphaFold predicted structures.

In [38]:
def create_graphs(
    train_structs_path, 
    test_structs_path, 
    target_train_seqs,
    target_test_seqs, 
    train_ref_seqs, 
    test_ref_seqs):

    train_output_dict = {}
    test_output_dict = {}
    
    for structs_path, seqs in zip([train_structs_path, test_structs_path], [train_ref_seqs, test_ref_seqs]):
        ds = structs_path[7:structs_path.find('_')]
        for f in tqdm(os.listdir(structs_path)):
            
            index = f[:f.find('_')]
            name = 'pnca_mut_' + ds + '_' + index

            pnca_m = pncaGraph(
                            pdb=f'{structs_path}/{f}',
                            lig_resname='PZA', 
                            self_loops=False,
                            cutoff_distance=12)
            
            metadata = seqs.iloc[[int(index)]]
            # display(metadata)
            
            mutation = metadata.mutation.values[0]

            if mutation in target_train_seqs.MUTATION.values:
                
                train_output_dict[name] = {
                    'graph':pnca_m, 
                    'metadata':metadata
                    }
            elif mutation in target_test_seqs.MUTATION.values:
                
                test_output_dict[name] = {
                    'graph':pnca_m, 
                    'metadata':metadata
                    }
            else:
                Exception('Mutation not found in clustered sequences.')
            
    return train_output_dict, test_output_dict

In [39]:
test_structs = '../pdb/test_pza'
train_structs = '../pdb/train_pza'
    

In [40]:
if split_method == 'clustered':
    train_graph_dict, test_graph_dict = create_graphs(
        train_structs, 
        test_structs, 
        clustered_train_seqs, 
        clustered_test_seqs, 
        train_seqs, 
        test_seqs)
    
if split_method == 'codon':
    train_graph_dict, test_graph_dict = create_graphs(
        train_structs, 
        test_structs, 
        codon_train_seqs, 
        codon_test_seqs, 
        train_seqs, 
        test_seqs)


100%|██████████| 200/200 [00:15<00:00, 12.71it/s]


In [41]:
# attach sequence from fasta to each key in dictionary
# iterate through dict and creat one sample with gen_dataset per graph
# assign dataset object to dictionary


for sample in tqdm(test_graph_dict):
    test_graph_dict[sample]['graph'].gen_dataset(
        sequences= test_graph_dict[sample]['metadata'],
        edge_weights= 'exp',
        lambda_param=2,
        normalise=True
        )

for sample in tqdm(train_graph_dict):
    
    train_graph_dict[sample]['graph'].gen_dataset(
        sequences= train_graph_dict[sample]['metadata'],
        edge_weights= 'exp',
        lambda_param=2,
        normalise=True
        )

 50%|████▉     | 99/200 [01:18<01:22,  1.22it/s]tri_norm: face with normal vector of lenght 0
tri_norm: face with normal vector of lenght 0
 86%|████████▌ | 398/464 [05:12<00:51,  1.28it/s]tri_norm: face with normal vector of lenght 0
tri_norm: face with normal vector of lenght 0
100%|██████████| 464/464 [06:04<00:00,  1.27it/s]


In [42]:
# check no nans in features

for sample in tqdm(test_graph_dict):
    assert torch.isnan(test_graph_dict[sample]['graph'].dataset[0].x).any() == False
    
for sample in tqdm(train_graph_dict):
    assert torch.isnan(train_graph_dict[sample]['graph'].dataset[0].x).any() == False

100%|██████████| 464/464 [00:00<00:00, 52913.46it/s]


In [43]:
# Create MinMax scaler fit only on training data
from sklearn.preprocessing import MinMaxScaler

train_features = [train_graph_dict[sample]['graph'].dataset[0].x for sample in train_graph_dict]
train_feats_cat = torch.cat([x for x in train_features], dim=0)

# fit scaler and save
scaler = MinMaxScaler()
scaler.fit(train_feats_cat.numpy())


with open(f"../data/{split_method}_scaler.pkl", "wb") as f:
    pkl.dump(scaler, f)

In [44]:
# apply scaler to train and test features

for sample in train_graph_dict:
    train_graph_dict[sample]['graph'].dataset[0].x = torch.tensor(
        scaler.transform(train_graph_dict[sample]['graph'].dataset[0].x.numpy()), 
        dtype=torch.float
    )
    
for sample in test_graph_dict:
    test_graph_dict[sample]['graph'].dataset[0].x = torch.tensor(
        scaler.transform(test_graph_dict[sample]['graph'].dataset[0].x.numpy()), 
        dtype=torch.float
    )

In [45]:
graph_dict = {
    'train': train_graph_dict,
    'test': test_graph_dict
}

In [46]:
# save graph_dict as pickle

with open(f'datasets/{split_method}_graph_dict.pkl', 'wb') as f:
    pkl.dump(graph_dict, f)